# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is accessible via a Croissant schema URL, providing structured metadata and links to survey records. The dataset follows AI-ready standards, facilitating transparent and reproducible workflows.

In [ ]:
# Install the mlcroissant library (uncomment if needed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessing metadata as object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and columns, referencing each by their `@id`.

Below, we enumerate the record sets and their fields from the dataset. Entities are identified by their `@id`.

In [ ]:
# List record sets and their fields by @id
record_sets = dataset.metadata.recordSet
if isinstance(record_sets, list):
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', '(unknown)')}")
            columns = field.get('column', [])
            if not isinstance(columns, list):
                columns = [columns]
            for col in columns:
                print(f"    Column @id: {col['@id']} (source: {col.get('source')})")
else:
    print("No record sets found.")

## 3. Data Extraction
Load records from specific record set(s) into DataFrames for analysis. Use the `@id` for each record set and field.

Below, we extract available survey data. If your dataset has multiple record sets, you may include their `@id`s in the list.

In [ ]:
# Get the list of recordSet @id's
record_set_ids = []
if isinstance(record_sets, list):
    for rs in record_sets:
        record_set_ids.append(rs['@id'])

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Extracted DataFrame for RecordSet @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())

# For demonstration, pick first record set id
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
    df = dataframes[primary_record_set_id]
else:
    primary_record_set_id = None
    df = pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data by key attributes. All references use `@id` as provided by the Croissant schema.

Below, we select a numeric field by its `@id` (e.g., `GAD_7_score`, `PHQ_9_score`, or similar), filter records above a threshold, normalize the values, and group the data by a demographic field (e.g., `level_of_education`, `gender`).

In [ ]:
# Try to find a numeric field (e.g., GAD-7 score) by @id
numeric_field_id = None
possible_numeric_fields = ['GAD_7_score', 'PHQ_9_score', 'PSQ_score', 'cr:GAD_7_score', 'cr:PHQ_9_score', 'cr:PSQ_score']
for col in df.columns:
    if col in possible_numeric_fields:
        numeric_field_id = col
        break

# Try to find a grouping field
group_field_id = None
possible_group_fields = ['level_of_education', 'gender', 'cr:level_of_education', 'cr:gender']
for col in df.columns:
    if col in possible_group_fields:
        group_field_id = col
        break

# Proceed only if a numeric field is found
if numeric_field_id:
    threshold = 10
    # Filter records
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a field
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please check column names.")

## 5. Visualization
Visualize the distribution of a numeric score and its relationship with a demographic grouping field. All references use field `@id`.

In [ ]:
# Visualize score distribution and group means
if numeric_field_id:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,4))
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        grouped.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to explore a Croissant dataset using the `mlcroissant` library. We loaded metadata and records using the Croissant schema URL, reviewed key entities by their `@id`, performed exploratory analysis on numeric fields, and visualized data distributions and groupings.

Key findings may include:
- Distribution of survey scores such as GAD-7, PHQ-9, or PSQ among Kilifi County residents
- Demographic correlations (e.g., education or gender) with mental health symptom scores

This workflow establishes best practices for AI-ready data exploration and reproducibility using Croissant standards.